In [ ]:
 # ============ Install dependencies (Colab) ============
!pip install dash dash-bootstrap-components pyngrok catboost pandas plotly
# ============ Imports ============
import os
import sys
import time
import base64
import io
import threading
import socket
from pyngrok import ngrok

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import joblib

# Plotly/Dash imports
import dash
from dash import Dash, html, dcc, dash_table, Input, Output, State
import dash_bootstrap_components as dbc
import plotly.express as px
import plotly.graph_objects as go

# ============ Mount Google Drive ============
from google.colab import drive
drive.mount('/content/drive')

# ============ Load Training and Test Data ============
try:
    train_path = "/content/drive/MyDrive/Enterprise_NIDS/UNSW_NB15_training-set.csv"
    Intrusion = pd.read_csv(train_path)
    test_path = "/content/drive/MyDrive/Enterprise_NIDS/UNSW_NB15_test_split.csv"
    Intrusion_test = pd.read_csv(test_path)
    print(f"Training data shape: {Intrusion.shape}")
    print(f"Test data shape: {Intrusion_test.shape}")
except FileNotFoundError as e:
    print("Error: Data files not found. Check your Google Drive paths.")
    raise e

# ============ Load Models ============

# LightGBM model and label encoder
lgb_path = '/content/drive/MyDrive/Enterprise_NIDS/Multiclass_lightgbm_model.pkl'
label_path = '/content/drive/MyDrive/Enterprise_NIDS/label_encoder.pkl'
try:
    model = joblib.load(lgb_path)
    label_encoder = joblib.load(label_path)
    print("Loaded LightGBM model and label encoder.")
except FileNotFoundError as e:
    print(f"Error: Model or encoder not found at {lgb_path} or {label_path}.")
    raise e

# ============ Preprocessing: Drop duplicates, handle missing, encode ============

# Drop duplicates
Intrusion.drop_duplicates(inplace=True)
Intrusion_test.drop_duplicates(inplace=True)

# Drop 'id' if present
if "id" in Intrusion.columns:
    Intrusion.drop(columns=["id"], inplace=True)
if "id" in Intrusion_test.columns:
    Intrusion_test.drop(columns=["id"], inplace=True)

# Fill numeric columns with median
for col in Intrusion.select_dtypes(include=[np.number]).columns:
    Intrusion[col] = pd.to_numeric(Intrusion[col], errors="coerce")
    Intrusion[col].fillna(Intrusion[col].median(), inplace=True)

for col in Intrusion_test.select_dtypes(include=[np.number]).columns:
    Intrusion_test[col] = pd.to_numeric(Intrusion_test[col], errors="coerce")
    Intrusion_test[col].fillna(Intrusion_test[col].median(), inplace=True)

# Ensure categorical columns are categories and fill missing
for col in ["proto", "service", "state"]:
    if col in Intrusion.columns:
        Intrusion[col] = Intrusion[col].fillna("missing").astype("category")
    if col in Intrusion_test.columns:
        Intrusion_test[col] = Intrusion_test[col].fillna("missing").astype("category")

# Prepare features and target
X = Intrusion.drop(columns=["attack_cat", "label"], errors="ignore")
y = label_encoder.transform(Intrusion["attack_cat"])
class_names = label_encoder.classes_
print(f"Classes: {class_names.tolist()}")

# Precompute predictions on test set for streaming
# Drop label/attack columns and 'id' if any
X_stream = Intrusion_test.drop(columns=["attack_cat", "label"], errors="ignore")
preds_numeric = model.predict(X_stream)
Intrusion_test["Prediction"] = label_encoder.inverse_transform(preds_numeric)
print("Precomputed predictions on test set for streaming.")

# ============ Build Static Figures ============

# Attack distribution from training data (static)
attack_distribution = Intrusion["attack_cat"].value_counts()
attack_fig = px.bar(
    x=attack_distribution.index,
    y=attack_distribution.values,
    title="Attack Distribution (Training Set)"
)
attack_fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#3E4B60",
    plot_bgcolor="#3E4B60",
    font=dict(color="white")
)

# Threat gauge (percentage of attacks in training set)
total_traffic = len(Intrusion)
normal_count = (Intrusion["attack_cat"] == "Normal").sum()
attack_count = total_traffic - normal_count
gauge_fig = go.Figure(go.Indicator(
    mode="gauge+number",
    value=100.0 * attack_count / total_traffic,
    title={"text": "Threat Percentage"},
    gauge={"axis": {"range": [0,100]}, "bar": {"color": "red"}}
))
gauge_fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#3E4B60",
    plot_bgcolor="#3E4B60",
    font=dict(color="white")
)

# ============ Initialize Dash App ============

app = Dash(__name__, external_stylesheets=[dbc.themes.DARKLY])

CARD_STYLE = {
    "backgroundColor": "#3E4B60",
    "border": "none",
    "borderRadius": "15px",
    "color": "white"
}

app.layout = dbc.Container([
    # Stores for predictions and info
    dcc.Store(id="stored-predictions", data=None),
    dcc.Store(id="stored-info", data=None),

    dcc.Interval(id="live-timer", interval=1*1000, n_intervals=0),

    html.H1("Enterprise Network Intrusion Intelligence System",
            className="text-center my-4", style={"color": "#83FBA9"}),

    # Tabs
    dbc.Tabs(active_tab="home", children=[

        # Home Tab
        dbc.Tab(label="🏠 Home", tab_id="home", children=[
            html.Br(),
            html.H3("Scanning the noise to catch the signal.", className="text-center text-info"),
            html.H5("Establishing the pattern of life; hunting the anomalies.", className="text-center text-info"),
            html.Hr(),
            html.H3("📋 Project Overview", className="text-center text-secondary"),
            html.P("AI-powered Enterprise Intrusion Detection System", className="text-center text-secondary"),
            html.H3("👤 Team Members", className="text-center text-secondary"),
            html.Ul([html.Li("Aadrika Rani"),
                     html.Li("Harnoor Kaur Sarao")
                     ],
                    style={
                        "listStyleType": "none",
                        "paddingLeft": "0",
                        "marginLeft": "0"
                        }, className="text-center text-secondary"),
            html.H3("🌐 Technology Stack", className="text-center text-secondary"),
            html.Ul([html.Li("Python"),
                     html.Li("Dash"),
                     html.Li("LightGBM"),
                     html.Li("numpy"),
                     html.Li("Pandas"),
                     html.Li("Plotly"),
                     html.Li("LocalTunnel"),
                     html.Li("Socket"),
                     html.Li("Google Colab Drive Mount"),
                     html.Li("Scikit-Learn"),
                     html.Li("Joblib"),
                     ],
                    style={
                        "listStyleType": "none",
                        "paddingLeft": "0",
                        "marginLeft": "0"
                        }, className="text-center text-secondary"),
            html.H3("🗂️ Dataset", className="text-center text-secondary"),
            html.Ul([
                html.Li("UNSW-NB15"),
                html.Li(f"Total rows (train): {len(Intrusion)}"),
                html.Li(f"Total rows (test): {len(Intrusion_test)}"),
                html.Li("Total Columns: 45 features"),
                html.Li("Sources: Kaggle"),
                ],
                    style={
                        "listStyleType": "none",
                        "paddingLeft": "0",
                        "marginLeft": "0"
                        },className="text-center text-secondary"),
            html.H3("🧮 Dataset Parameters", className="text-center text-secondary"),
            html.Ul([
                html.Li("Outputs: Multi-Class 'attack_cat' into 9 categories"),
                html.Li("Categories: Generic, Exploits, Fuzzers, DoS, Reconnaissance, Analysis, Backdoors, Shellcode, Worms",)
            ],
            style={
                "listStyleType": "none",
                "paddingLeft": "0",
                "marginLeft": "0"
                },className="text-center text-secondary")
        ]),

        # SOC Dashboard Tab
        dbc.Tab(label="📊 SOC Dashboard", tab_id="dashboard", children=[
            dbc.Row([
                dbc.Col(dbc.Card(dbc.CardBody([
                    html.H4("Total Packets"),
                    html.H2(id="total-card", children=0)
                ]), style=CARD_STYLE)),
                dbc.Col(dbc.Card(dbc.CardBody([
                    html.H4("Normal"),
                    html.H2(id="normal-card", children=0)
                ]), style=CARD_STYLE)),
                dbc.Col(dbc.Card(dbc.CardBody([
                    html.H4("Attacks"),
                    html.H2(id="attack-card", children=0)
                ]), style=CARD_STYLE))
            ]),
            html.Br(),
            dbc.Row([
                dbc.Col(dcc.Graph(id="attack-graph", figure=attack_fig), width=8),
                dbc.Col(dcc.Graph(id="gauge-graph", figure=gauge_fig), width=4)
            ])
        ]),

        # Live Monitor Tab
        dbc.Tab(label="📡 Live Monitor", tab_id="live", children=[
            html.H3("Live Packet Monitor"),
            html.Div(id="live-monitor", children="")
        ]),

        # AI Prediction Tab
        dbc.Tab(label="🤖 AI Prediction", tab_id="ai", children=[
            html.H3("AI-Based Intrusion Detection"),
            html.P("Upload a network traffic CSV file for prediction."),
            dcc.Upload(
                id="upload-data",
                children=html.Div(["📂 Drag & Drop or ", html.A("Select CSV File")]),
                style={
                    "width": "100%", "height": "80px", "lineHeight": "80px",
                    "borderWidth": "2px", "borderStyle": "dashed", "borderRadius": "10px",
                    "textAlign": "center", "marginBottom": "20px",
                    "backgroundColor": "#1F2A40", "color": "white"
                },
                multiple=False
            ),
            html.Div(id="selected-file", style={"color": "lightgreen", "marginTop": "10px"}),
            html.Br(),

            dbc.Card(dbc.CardBody([
                html.H5("Upload Status"),
                dbc.Progress(id="upload-progress", value=0, striped=True, animated=True, style={"height":"25px"}),
                html.Br(),
                html.Div(id="upload-status"),
                html.Hr(),
                html.Div(id="file-info")
            ])),
            html.Br(),

            dbc.Button("▶ Run LightGBM Prediction", id="predict-btn", color="success"),
            html.Br(), html.Br(),

            html.Div(id="prediction-output"),

        ]),

        # About Tab
        dbc.Tab(label="ℹ️ About", tab_id="about", children=[
            html.H3("🗃️ Dataset",className="text-center text-secondary"),
            html.H5("UNSW-NB15", className="text-center text-secondary"),
            html.H3("⚙️ Models",className="text-center text-secondary"),
            html.H5("LightGBM", className="text-center text-secondary"),
            html.H3("📈 Model Performance",className="text-center text-secondary"),
            html.H5("Accuracy: 82.67%", className="text-center text-secondary"),
            html.H3("👥 Authors",className="text-center text-secondary"),
            html.H5("Harnoor Kaur Sarao & Aadrika Rani", className="text-center text-secondary")
        ])
    ])
], fluid=True, style={"backgroundColor": "#0A1930", "minHeight": "100vh", "padding": "20px"})

# ============ Callbacks ============

# Show selected filename
@app.callback(
    Output("selected-file", "children"),
    Input("upload-data", "filename")
)
def show_filename(filename):
    if not filename:
        return "No file selected."
    return f"Selected File: {filename}"

# Upload & Predict CSV
@app.callback(
    Output("stored-predictions", "data"),
    Output("prediction-output", "children"),
    Output("upload-progress", "value"),
    Output("upload-status", "children"),
    Output("file-info", "children"),
    Input("predict-btn", "n_clicks"),
    State("upload-data", "contents"),
    State("upload-data", "filename"),
    prevent_initial_call=True
)
def predict_csv(n_clicks, contents, filename):
    if contents is None:
        # No file uploaded
        return (None,
                dbc.Alert("Please upload a CSV file first.", color="warning"),
                0, "No file uploaded.", "")
    # Update progress to 50%
    progress = 50
    try:
        content_type, content_string = contents.split(',')
        decoded = base64.b64decode(content_string)
        df = pd.read_csv(io.StringIO(decoded.decode("utf-8")))
    except Exception as e:
        return (None,
                dbc.Alert(f"Error parsing file: {e}", color="danger"),
                progress, "Failed to parse CSV.", "")
    # Preprocess like training: drop unwanted cols
    X_new = df.copy().drop(columns=["attack_cat", "label"], errors="ignore")
    if "id" in X_new.columns:
        X_new.drop(columns=["id"], inplace=True)
    for col in ["proto", "service", "state"]:
        if col in X_new.columns:
            X_new[col] = X_new[col].astype(str).fillna("missing").astype("category")
    # Predict
    preds = model.predict(X_new)
    df["Prediction"] = label_encoder.inverse_transform(preds)
    # Prepare outputs
    memory = round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    file_info = dbc.Card(dbc.CardBody([
        html.H5("Uploaded File"),
        html.Hr(),
        html.P(f"Filename: {filename}"),
        html.P(f"Rows: {len(df):,}"),
        html.P(f"Columns: {len(df.columns)}"),
        html.P(f"Memory: {memory} MB"),
        html.P("Prediction Complete", style={"color": "lime"})
    ]))
    table = dash_table.DataTable(
        data=df.head(20).to_dict("records"),
        columns=[{"name": i, "id": i} for i in df.columns],
        page_size=10,
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "center", "backgroundColor": "#222", "color": "white"},
        style_header={"backgroundColor": "#111", "fontWeight": "bold"}
    )
    # Return stored predictions JSON and UI elements
    return (
        df.to_json(date_format='iso', orient='split'),
        table,
        100,
        dbc.Alert("Prediction completed successfully.", color="success"),
        file_info
    )



# SOC Dashboard Cards
@app.callback(
    Output("total-card", "children"),
    Output("normal-card", "children"),
    Output("attack-card", "children"),
    Input("live-timer", "n_intervals"),
    Input("stored-predictions", "data")
)
def update_cards(n, data_json):
    PACKETS_PER_SEC = 500
    if data_json is not None:
        df = pd.read_json(data_json, orient="split")
    else:
        df = Intrusion_test  # Use preloaded test data if no upload
    current = min((n + 1) * PACKETS_PER_SEC, len(df))
    partial = df.iloc[:current]
    total = len(partial)
    normal = (partial["Prediction"] == "Normal").sum()
    attack = total - normal
    return total, normal, attack

# Gauge Update
@app.callback(
    Output("gauge-graph", "figure"),
    Input("live-timer", "n_intervals"),
    Input("stored-predictions", "data")
)
def update_gauge(n, data_json):
    PACKETS_PER_SEC = 500
    if data_json is not None:
        df = pd.read_json(data_json, orient="split")
    else:
        df = Intrusion_test
    current = min((n + 1) * PACKETS_PER_SEC, len(df))
    partial = df.iloc[:current]
    if current > 0:
        percent = 100.0 * (partial["Prediction"] != "Normal").sum() / current
    else:
        percent = 0
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=percent,
        title={"text": "Threat Percentage"},
        gauge={"axis": {"range":[0,100]}, "bar":{"color":"red"}}
    ))
    fig.update_layout(template="plotly_dark",
                      paper_bgcolor="#3E4B60", plot_bgcolor="#3E4B60",
                      font=dict(color="white"))
    return fig

# Attack Distribution Graph
@app.callback(
    Output("attack-graph", "figure"),
    Input("live-timer", "n_intervals"),
    Input("stored-predictions", "data")
)
def update_attack_graph(n, data_json):
    PACKETS_PER_SEC = 500
    if data_json is not None:
        df = pd.read_json(data_json, orient="split")
    else:
        df = Intrusion_test
    current = min((n + 1) * PACKETS_PER_SEC, len(df))
    partial = df.iloc[:current]
    counts = partial["Prediction"].value_counts()
    fig = px.bar(x=counts.index, y=counts.values, title="Live Attack Distribution")
    fig.update_layout(template="plotly_dark",
                      paper_bgcolor="#3E4B60", plot_bgcolor="#3E4B60",
                      font=dict(color="white"))
    return fig

#live monitor
@app.callback(
    Output("live-monitor", "children"),
    Input("live-timer", "n_intervals"),
    Input("stored-predictions", "data")
)
def update_live_monitor(n_intervals, data_json):
    # Number of packets to simulate per interval
    PACKETS_PER_SEC = 1  # adjust as needed
    # Load data from upload or fallback to test set
    if data_json is not None:
        df = pd.read_json(data_json, orient="split")
    else:
        df = Intrusion_test

    # Determine how many packets to show so far
    current = min((n_intervals + 1) * PACKETS_PER_SEC, len(df))
    partial = df.iloc[:current]

    # Take the last N packets for display
    N = 5  # show last 5 packets
    if current > 0:
        display_packets = partial.tail(N)
    else:
        display_packets = pd.DataFrame(columns=partial.columns)

    # Build cards for each packet
    cards = []
    for idx, row in display_packets.iterrows():
        status = row["Prediction"]
        if status == "Normal":
            # Normal packet: green card
            bg_color = "#4caf50"
            card_text = "Normal"
        else:
            # Attack packet: red card, include attack category
            bg_color = "#f44336"
            card_text = f"Attack: {status}"

        card = dbc.Card(
            dbc.CardBody(html.H4(card_text, style={"color": "white"})),
            style={"backgroundColor": bg_color, "color": "white", "marginBottom": "10px"}
        )
        cards.append(card)

    # Return the list of cards to render
    return cards

# ============ Run Dash with ngrok (for Colab) ============

import socket
import threading

def get_free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

PORT = get_free_port()

def run_app():
    app.run(
        host="0.0.0.0",
        port=PORT,
        debug=False
    )

thread = threading.Thread(target=run_app)
thread.daemon = True
thread.start()

print(f"Dashboard started on port {PORT}")

import threading
import os
import time

# 1. Choose a fixed port
PORT = 8050

def run_app():
    app.run(
        host="0.0.0.0",
        port=PORT,
        debug=False
    )

# 2. Start the Dash app in a background thread
thread = threading.Thread(target=run_app)
thread.daemon = True
thread.start()

# Give the app a moment to start up
time.sleep(2)

# 3. Install localtunnel and expose the port
print("--- Creating Public Tunnel ---")
!npm install -g localtunnel
!lt --port {PORT}


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.6 MB/s eta 0:00:00
Mounted at /content/drive
Training data shape: (175341, 45)
Test data shape: (35069, 43)
Loaded LightGBM model and label encoder.


/tmp/ipykernel_2672/2333782280.py:69: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  Intrusion[col].fillna(Intrusion[col].median(), inplace=True)
/tmp/ipykernel_2672/2333782280.py:73: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplac

Classes: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']
Precomputed predictions on test set for streaming.
Dash is running on http://0.0.0.0:37185/



INFO:dash.dash:Dash is running on http://0.0.0.0:37185/



Dashboard started on port 37185
Dash is running on http://0.0.0.0:8050/



INFO:dash.dash:Dash is running on http://0.0.0.0:8050/



 * Serving Flask app '__main__'
 * Serving Flask app '__main__'
 * Debug mode: off
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:37185
 * Running on http://172.28.0.12:37185
INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8050
 * Running on http://172.28.0.12:8050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:Press CTRL+C to quit


--- Creating Public Tunnel ---
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 2s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.1
npm notice To update run: npm install -g npm@12.0.1
npm notice
⠸your url is: https://loud-regions-cheer.loca.lt


INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 11:48:18] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 11:48:18] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_4_0m1783943237.12.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 11:48:19] "GET /_dash-component-suites/dash/deps/react@18.v4_4_0m1783943237.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 11:48:19] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_4_0m1783943237.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 11:48:19] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_4_0m1783943237.8.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 11:48:19] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_4_0m1783943237.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 11:48:19] "GET /_dash-component-suites/dash/dcc/dash_core_components.v4_4_0m1783943237.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13